# Video DS Playground — Интерактивный анализ

Этот ноутбук позволяет выполнять этапы пайплайна пошагово и исследовать промежуточные результаты.

In [ ]:
import sys
sys.path.insert(0, '..')  # чтобы импортировать src/

# Регистрируем все компоненты
import src.stages
import src.models
from src.core.registry import list_stages, list_models

print('Доступные этапы:', list_stages())
print('Доступные модели:', list_models())

## 1. Извлечение аудио

In [ ]:
from src.utils.ffmpeg_helper import extract_audio, get_media_info

VIDEO_PATH = '../data/raw_videos/sample.mp4'  # укажите свой файл
AUDIO_PATH = '../data/audio/sample_16k.wav'

audio_path = extract_audio(VIDEO_PATH, AUDIO_PATH)
meta = get_media_info(audio_path)
print('Метаданные аудио:', meta)

## 2. Визуализация аудиограммы

In [ ]:
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np

data, sr = sf.read(AUDIO_PATH)
times = np.linspace(0, len(data)/sr, len(data))

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(times, data, alpha=0.7, linewidth=0.5)
ax.set_xlabel('Время (сек)')
ax.set_ylabel('Амплитуда')
ax.set_title('Аудиограмма')
plt.tight_layout()
plt.show()
print(f'Длительность: {len(data)/sr:.1f} сек | Sample rate: {sr} Гц')

## 3. VAD — детекция речи

In [ ]:
# TODO: Реализуйте SileroVADWrapper.load() и predict() (задание 05)
# from src.core.registry import load_model
# vad_model = load_model('silero')
# vad_result = vad_model.predict(AUDIO_PATH)
# speech_intervals = vad_result['speech_intervals']
# print(f'Речевых сегментов: {len(speech_intervals)}')
# print('Первые 5:', speech_intervals[:5])

## 4. Транскрибация (Whisper)

In [ ]:
from src.core.registry import load_model

# Загружаем модель
whisper = load_model('whisper-small', device='cpu')

# Транскрибируем
result = whisper.predict(AUDIO_PATH, language='ru')
print('Текст:', result['text'][:500])
print(f"Язык: {result['language']}")
print(f"Сегментов: {len(result['segments'])}")

## 5. Сравнение WER нескольких моделей

In [ ]:
import pandas as pd
from src.utils.metrics import compute_wer

# Укажите эталонный текст
REFERENCE = 'Ваш эталонный текст здесь'

MODELS = ['whisper-tiny', 'whisper-small']
rows = []
for model_name in MODELS:
    model = load_model(model_name, device='cpu')
    res = model.predict(AUDIO_PATH, language='ru')
    wer = compute_wer(REFERENCE, res['text'])
    rows.append({'model': model_name, 'wer': round(wer, 4),
                 'time_sec': res.get('inference_time_sec', 0)})

df = pd.DataFrame(rows).sort_values('wer')
print(df.to_string(index=False))

## 6. Полный пайплайн через Pipeline

In [ ]:
from src.core.pipeline import Pipeline

pipeline = Pipeline('../config/default_pipeline.yaml')
print(pipeline.describe())

# Запуск
# result = pipeline.run(VIDEO_PATH)
# print('Транскрипция:', result.get('clean_transcription', '')[:300])